# 混合检索与 Rerank：LlamaIndex

流程：LlamaIndex 的 BM25 Retriever + Vector Retriever → 合并候选节点 → Cross-Encoder Reranker 重排序。

In [ ]:
# 如果环境中没有相关依赖，可先执行：
# %pip install llama-index llama-index-retrievers-bm25 llama-index-llms-google-genai llama-index-embeddings-huggingface sentence-transformers

import os
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.schema import QueryBundle
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.retrievers.bm25 import BM25Retriever

load_dotenv()
Settings.llm = GoogleGenAI(
    model="gemini-3.1-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)
Settings.embed_model = HuggingFaceEmbedding(model_name="moka-ai/m3e-base")
reranker = CrossEncoder("BAAI/bge-reranker-base")

In [ ]:
documents = SimpleDirectoryReader(
    input_dir="../knowledge_db/prompt_engineering",
    recursive=True,
).load_data()
index = VectorStoreIndex.from_documents(documents)
vector_retriever = index.as_retriever(similarity_top_k=8)
bm25_retriever = BM25Retriever.from_defaults(
    docstore=index.docstore,
    similarity_top_k=8,
)

In [ ]:
question = "总结文本转换这篇文章的主要观点、方法和示例"
query_bundle = QueryBundle(question) # 把普通的问题字符串包装成 LlamaIndex 的查询对象。
bm25_nodes = bm25_retriever.retrieve(query_bundle)
vector_nodes = vector_retriever.retrieve(query_bundle)

# 合并并按节点 ID 去重
nodes = {item.node.node_id: item for item in bm25_nodes + vector_nodes}
candidate_nodes = list(nodes.values())
pairs = [(question, item.node.get_content()) for item in candidate_nodes]
scores = reranker.predict(pairs)
ranked_nodes = [
    item for _, item in sorted(
        zip(scores, candidate_nodes),
        key=lambda pair: pair[0],
        reverse=True,
    )
]
final_nodes = ranked_nodes[:5]

for score, item in sorted(zip(scores, candidate_nodes), reverse=True, key=lambda pair: pair[0])[:5]:
    print(round(float(score), 4), item.node.metadata.get("file_path"))

`query_bundle = QueryBundle(question)`

```py
QueryBundle(
    query_str="什么是文本转换？"
)
```

In [ ]:
context = "\n\n".join(item.node.get_content() for item in final_nodes)
response = Settings.llm.complete(
    f"只根据上下文回答问题，覆盖多个相关片段，不要只总结一个示例。\n\n上下文：{context}\n问题：{question}"
)
print(response)